In [ ]:
# start coding here

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path



In [ ]:
# Load bimodal_bridge model CLIP scores
bimodal_bridge_model = snakemake.params.model_mappings["bimodal_bridge"]

df = pd.read_csv(snakemake.input.model_results)

# %%
# Analyze and rank samples by absolute CLIP scores
df["avg_clip_score"] = (df["clip_score_left_right"] + df["clip_score_right_left"]) / 2

sample_analysis = (
    df.groupby("orig_ids")
    .agg(
        {
            "avg_clip_score": "mean",
            "clip_score_left_right": "mean",
            "clip_score_right_left": "mean",
        }
    )
    .reset_index()
)

sample_analysis["clip_score_rank"] = sample_analysis["avg_clip_score"].rank(
    ascending=False
)
sample_analysis = sample_analysis.sort_values("avg_clip_score", ascending=False)

In [ ]:
# %%
# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(20, 16))
fig.suptitle(
    f"Bimodal Bridge Model: Per-Sample CLIP Score Analysis\nTest Dataset: {snakemake.wildcards.dataset}",
    fontsize=16,
)

# Top 10 samples
top_samples = sample_analysis.head(10)
axes[0, 0].barh(
    range(len(top_samples)), top_samples["avg_clip_score"], color="green", alpha=0.7
)
axes[0, 0].set_yticks(range(len(top_samples)))
axes[0, 0].set_yticklabels(
    [f"Sample {row['orig_ids']}" for _, row in top_samples.iterrows()]
)
axes[0, 0].set_xlabel("Average CLIP Score")
axes[0, 0].set_title("Top 10 Samples")
axes[0, 0].grid(True, alpha=0.3)

# Bottom 10 samples
bottom_samples = sample_analysis.tail(10)
axes[0, 1].barh(
    range(len(bottom_samples)),
    bottom_samples["avg_clip_score"],
    color="red",
    alpha=0.7,
)
axes[0, 1].set_yticks(range(len(bottom_samples)))
axes[0, 1].set_yticklabels(
    [f"Sample {row['orig_ids']}" for _, row in bottom_samples.iterrows()]
)
axes[0, 1].set_xlabel("Average CLIP Score")
axes[0, 1].set_title("Bottom 10 Samples")
axes[0, 1].grid(True, alpha=0.3)

# Directional comparison
axes[1, 0].scatter(
    sample_analysis["clip_score_left_right"],
    sample_analysis["clip_score_right_left"],
    alpha=0.6,
)
min_val = min(
    sample_analysis["clip_score_left_right"].min(),
    sample_analysis["clip_score_right_left"].min(),
)
max_val = max(
    sample_analysis["clip_score_left_right"].max(),
    sample_analysis["clip_score_right_left"].max(),
)
axes[1, 0].plot([min_val, max_val], [min_val, max_val], "r--", alpha=0.8)
axes[1, 0].set_xlabel("Left-Right CLIP Score")
axes[1, 0].set_ylabel("Right-Left CLIP Score")
axes[1, 0].set_title("Directional Comparison")
axes[1, 0].grid(True, alpha=0.3)

# Distribution
axes[1, 1].hist(sample_analysis["avg_clip_score"], bins=20, alpha=0.7, color="blue")
axes[1, 1].axvline(
    sample_analysis["avg_clip_score"].mean(),
    color="green",
    linestyle="-",
    alpha=0.8,
    label="Mean",
)
axes[1, 1].axvline(
    sample_analysis["avg_clip_score"].median(),
    color="orange",
    linestyle="--",
    alpha=0.8,
    label="Median",
)
axes[1, 1].set_xlabel("Average CLIP Score")
axes[1, 1].set_ylabel("Number of Samples")
axes[1, 1].set_title("Score Distribution")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(snakemake.output.plot)

In [ ]:
print(f"=== BIMODAL BRIDGE MODEL SUMMARY ({snakemake.wildcards.dataset}) ===")
print(f"Total samples: {len(sample_analysis)}")
print(f'Mean CLIP score: {sample_analysis["avg_clip_score"].mean():.4f}')
print(
    f'Score range: {sample_analysis["avg_clip_score"].min():.4f} - {sample_analysis["avg_clip_score"].max():.4f}'
)

sample_analysis.to_csv(snakemake.output.analysis, index=False)